In [3]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 2.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 41.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 40.6 MB/s eta 0:00:0000:0100:01m


In [1]:
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce


from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/Posdoc/Native_AFP/code/')
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import GCNModel, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display

import itertools
from sklearn.metrics import r2_score
import scipy


import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [2]:
def prepare_model_and_data(raw_filename, target_name='Calx', targets=None, random_seed=888, batch_size=50, epochs=800, p_dropout=0.1, fingerprint_dim=210, weight_decay=5, learning_rate=3, radius=4, T=2, per_target_output_units_num=1):
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    feature_filename = raw_filename.replace('.csv', '.pickle')
    filename = raw_filename.replace('.csv', '')
    prefix_filename = raw_filename.split('/')[-1].replace('.csv', '')
    smiles_targets_df = pd.read_csv(raw_filename)

    smilesList = smiles_targets_df.SMILES.values
    print("number of all smiles: ", len(smilesList))

    atom_num_dist = []
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            atom_num_dist.append(len(mol.GetAtoms()))
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            print(smiles)
            pass
    print("number of successfully processed smiles: ", len(remained_smiles))

    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    start_time = str(time.ctime()).replace(':', '-').replace(' ', '_')
    output_units_num = len(targets) * per_target_output_units_num

    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = save_smiles_dicts(smilesList, filename)
    feature_dicts = get_smiles_dicts(smilesList)

    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array([canonical_smiles_list[0]], feature_dicts)
    num_atom_features = x_atom.shape[-1]
    num_bond_features = x_bonds.shape[-1]

    loss_function = nn.MSELoss()
    model = GCNModel(radius, T, num_atom_features, num_bond_features, fingerprint_dim, output_units_num, p_dropout)
    model.to(device)

    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    #print(params)
    #for name, param in model.named_parameters():
     #   if param.requires_grad:
      #      print(name, param.data.shape)

    # Create log_df
    log_df = pd.DataFrame(np.log(remained_df[targets]), columns=targets, index=remained_df.index)

    # Add the remaining columns
    log_df = pd.concat([log_df, remained_df[['SMILES', 'Host','cano_smiles']]], axis=1)


    return model, optimizer, loss_function, remained_df,log_df, targets, feature_dicts



In [3]:
raw_filename= '/notebooks/data/Data excluding non-binders.csv'
model, optimizer, loss_function, remained_df,log_df, targets, feature_dicts= prepare_model_and_data(raw_filename)

number of all smiles:  40
number of successfully processed smiles:  40


In [3]:
def train(model, dataset, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list):
    model.train()
    np.random.seed(epoch)
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]

    else:
        valList = np.arange(0, dataset.shape[0])
        np.random.shuffle(valList)
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    total_loss = 0
    
    for counter, batch in enumerate(batch_list):
        batch_df = dataset.loc[batch, :]
        smiles_list = batch_df.cano_smiles.values

        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(smiles_list, feature_dicts)
        
        x_atom = torch.Tensor(x_atom).to(device)
        #print(x_atom.shape)
        x_bonds = torch.Tensor(x_bonds).to(device)
        x_atom_index = torch.LongTensor(x_atom_index).to(device)
        x_bond_index = torch.LongTensor(x_bond_index).to(device)
        x_mask = torch.Tensor(x_mask).to(device)
        
        atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
        
        optimizer.zero_grad()
        loss = 0.0
        
        for i, target in enumerate(targets):
            y_pred = mol_prediction[:, i]
            y_val = torch.Tensor(batch_df[target].values).squeeze().to(device)
            target_loss = loss_function(y_pred, y_val) * (ratio_list[i] ** 2)
            loss += target_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(batch_list)


def eval(model, dataset, targets, feature_dicts, batch_size):
    model.eval()
    
    eval_MAE_list = {target: [] for target in targets}
    eval_MSE_list = {target: [] for target in targets}
    y_val_list = {target: [] for target in targets}
    y_pred_list = {target: [] for target in targets}
    smiles_list = []
    
    if len(dataset) <= batch_size:
        batch_list = [dataset.index]
    else:
        valList = np.arange(0, dataset.shape[0])
        batch_list = [valList[i:i+batch_size] for i in range(0, dataset.shape[0], batch_size)]
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for batch in batch_list:
            batch_df = dataset.loc[batch, :]
            batch_smiles = batch_df.cano_smiles.values
            smiles_list.extend(batch_smiles)
            
            x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, _ = get_smiles_array(batch_smiles, feature_dicts)
            
            x_atom = torch.Tensor(x_atom).to(device)
            x_bonds = torch.Tensor(x_bonds).to(device)
            x_atom_index = torch.LongTensor(x_atom_index).to(device)
            x_bond_index = torch.LongTensor(x_bond_index).to(device)
            x_mask = torch.Tensor(x_mask).to(device)
            
            atoms_prediction, mol_prediction = model(x_atom, x_bonds, x_atom_index, x_bond_index, x_mask)
            
            for i, target in enumerate(targets):
                y_pred = mol_prediction[:, i].view(-1, 1)
                y_val = torch.Tensor(batch_df[target].values.reshape(-1, 1)).to(device)
                
                MAE = F.l1_loss(y_pred, y_val, reduction='none')
                MSE = F.mse_loss(y_pred, y_val, reduction='none')
                
                y_pred_list[target].extend(y_pred.cpu().numpy().flatten())
                y_val_list[target].extend(y_val.cpu().numpy().flatten())
                
                eval_MAE_list[target].extend(MAE.cpu().numpy().flatten())
                eval_MSE_list[target].extend(MSE.cpu().numpy().flatten())
    
    eval_MAE = np.array([np.mean(eval_MAE_list[target]) for target in targets])
    eval_MSE = np.array([np.mean(eval_MSE_list[target]) for target in targets])
    
    return eval_MAE, eval_MSE, y_pred_list, y_val_list, smiles_list




import numpy as np
from collections import defaultdict
from sklearn.model_selection import LeaveOneOut

def train_and_evaluate(model, remained_df, targets, feature_dicts, optimizer, loss_function, batch_size, num_epochs, patience=20, min_delta=0.001, prefix_filename='', start_time=''):
    best_param = {"train_epoch": 0, "test_epoch": 0, "train_MSE": float('inf'), "test_MSE": float('inf')}
    stats_list = []
    ratio_list = [1.0] * len(targets)
    loo = LeaveOneOut()
    fold_results = []
    best_fold_results = []
    regression = {}

    train_predictions = {target: [] for target in targets}
    train_actuals = {target: [] for target in targets}
    test_predictions = {target: [] for target in targets}
    test_actuals = {target: [] for target in targets}
    
    initial_state = {k: v.clone() for k, v in model.state_dict().items()}
    for fold, (train_index, test_index) in enumerate(loo.split(remained_df)):
        print(f"\nFold {fold + 1}/{len(remained_df)}")

        # Reset model parameters to initial state
        model.load_state_dict(initial_state)
        
        # Reset optimizer state
        optimizer.state = defaultdict(dict)

        train_df = remained_df.iloc[train_index].copy()
        test_df = remained_df.iloc[test_index].copy()
        current_smile = test_df['Host'].values[0]

        best_train_mse = float('inf')
        best_model_state = None
        best_val_mse = float('inf')
        epochs_no_improve = 0
        best_train_metrics = None

        for epoch in range(num_epochs):
            train_loss = train(model, train_df, optimizer, loss_function, epoch, batch_size, targets, feature_dicts, ratio_list)
            train_MAE, train_MSE, train_pred, train_actual, _ = eval(model, train_df, targets, feature_dicts, batch_size)

            if train_MSE.mean() < best_train_mse:
                best_train_mse = train_MSE.mean()
                best_model_state = model.state_dict()
                best_param["train_epoch"] = epoch
                best_param["train_MSE"] = train_MSE.mean()
                best_train_metrics = (train_MAE, train_MSE, train_pred, train_actual)

            # Evaluate on validation set
            val_MAE, val_MSE, val_pred, val_actual, _ = eval(model, test_df, targets, feature_dicts, batch_size)

            if val_MSE.mean() < best_val_mse - min_delta:
                best_val_mse = val_MSE.mean()
                epochs_no_improve = 0
                best_model_state = model.state_dict()
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

        # Load the best model state for this fold
        model.load_state_dict(best_model_state)

        # Use the best train metrics
        train_MAE, train_MSE, train_pred, train_actual = best_train_metrics

        # Save train metrics only for the best epoch
        for target in targets:
            train_predictions[target].extend(train_pred[target])
            train_actuals[target].extend(train_actual[target])

        # Evaluate on test set only once, after training is complete
        test_MAE, test_MSE, test_pred, test_actual, test_smiles = eval(model, test_df, targets, feature_dicts, batch_size)

        for target in targets:
            test_predictions[target].extend(test_pred[target])
            test_actuals[target].extend(test_actual[target])

        best_fold_results.append((train_MAE.mean(), train_MSE.mean(), test_MAE.mean(), test_MSE.mean()))
        
        # Create the dictionary entry for the current test row
        regression[current_smile] = {}
        for target in targets:
            regression[current_smile][target] = {
                "predicted": test_pred[target][0],
                "actual": test_actual[target][0]
            }

        print(f"\nTest point details for Fold {fold + 1}:")
        print(f"SMILES: {current_smile}")
        for target in targets:
            print(f"{target}:")
            print(f" Actual value: {test_actual[target][0]:.4f}")
            print(f" Predicted value: {test_pred[target][0]:.4f}")

        fold_results.append({
            'fold': fold + 1,
            'smiles': current_smile,
            'actual_values': {target: test_actual[target][0] for target in targets},
            'predicted_values': {target: test_pred[target][0] for target in targets}
        })

    overall_train_mae = np.mean([res[0] for res in best_fold_results])
    overall_train_mse = np.mean([res[1] for res in best_fold_results])
    overall_test_mae = np.mean([res[2] for res in best_fold_results])
    overall_test_mse = np.mean([res[3] for res in best_fold_results])

    print("\nFinal Results:")
    print(f"Overall Training MAE: {overall_train_mae:.4f}")
    print(f"Overall Training MSE: {overall_train_mse:.4f}")
    print(f"Overall Test MAE: {overall_test_mae:.4f}")
    print(f"Overall Test MSE: {overall_test_mse:.4f}")

    return overall_train_mae, overall_train_mse, overall_test_mae, overall_test_mse, fold_results, regression, train_predictions, train_actuals, test_predictions, test_actuals

In [7]:

model, optimizer, loss_function, remained_df,log_df, targets, feature_dicts = prepare_model_and_data('/notebooks/data/Data excluding non-binders.csv')
"""overall_train_mae, overall_train_mse, overall_valid_mae, overall_valid_mse, fold_results, regression = train_and_evaluate(
    model, 
    remained_df, 
    targets, 
    feature_dicts, 
    optimizer, 
    loss_function, 
    batch_size=50, 
    num_epochs=5, 
    prefix_filename='your_prefix', 
    start_time='your_start_time',
  # Ensure scaler is passed here
)"""


results = train_and_evaluate(model, log_df, targets, feature_dicts, optimizer, loss_function, batch_size=50, num_epochs=800)
train_predictions, train_actuals, test_predictions, test_actuals = results[-4:]


                                                                                 

number of all smiles:  40
number of successfully processed smiles:  40

Fold 1/40
Early stopping at epoch 29

Test point details for Fold 1:
SMILES: PSC4
H3K4:
 Actual value: 1.4110
 Predicted value: 0.7888
H3K4ac:
 Actual value: 2.8332
 Predicted value: 1.0794
H3K4me1:
 Actual value: 0.7885
 Predicted value: -0.2256
H3K4me2:
 Actual value: -0.1863
 Predicted value: -1.4551
H3K4me3:
 Actual value: -0.5798
 Predicted value: -1.4944
H3K9me3:
 Actual value: 0.3365
 Predicted value: -0.8149
H3R2me2a:
 Actual value: 0.8329
 Predicted value: -0.5400
H3R2me2s:
 Actual value: 1.7405
 Predicted value: 0.1285

Fold 2/40
Early stopping at epoch 26

Test point details for Fold 2:
SMILES: PSC6
H3K4:
 Actual value: -0.9676
 Predicted value: 1.5911
H3K4ac:
 Actual value: 1.0647
 Predicted value: 2.1331
H3K4me1:
 Actual value: -1.3471
 Predicted value: -0.1422
H3K4me2:
 Actual value: -2.8473
 Predicted value: -2.4798
H3K4me3:
 Actual value: -4.0745
 Predicted value: -2.4200
H3K9me3:
 Actual value: -2.

In [9]:
import numpy as np
from sklearn.metrics import r2_score

def calculate_overall_r2(data):
    all_predicted = []
    all_actual = []
    
    for compound in data.values():
        for histone_mod in compound.values():
            all_predicted.append(histone_mod['predicted'])
            all_actual.append(histone_mod['actual'])

    all_predicted = np.array(all_predicted)
    all_predicted = np.exp(all_predicted)
    
    all_actual = np.array(all_actual)
    all_actual = np.exp(all_actual)
    overall_r2 = r2_score(all_actual, all_predicted)
    return overall_r2

# Assuming your dictionary is stored in a variable called 'data'
overall_r2 = calculate_overall_r2(results[5])

print(f"Overall R² score: {overall_r2:.4f}")


Overall R² score: -0.0079


In [6]:
log_df

,H3K4,H3K4ac,H3K4me1,H3K4me2,H3K4me3,H3K9me3,H3R2me2a,H3R2me2s,SMILES,Host,cano_smiles
0,1.410987,2.833213,0.788457,-0.186330,-0.579818,0.336472,0.832909,1.740466,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,PSC4,O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...
1,-0.967584,1.064711,-1.347074,-2.847312,-4.074542,-2.476938,-1.514128,-0.994252,OC1=C2C=C(S(=O)(O)=O)C=C1CC3=C(O)C(CC4=CC(S(=O...,PSC6,O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...
2,2.833213,2.785011,2.028148,0.587787,0.336472,1.887070,1.098612,0.741937,OC1=C(CC2=CC([N+]([O-])=O)=CC(C3)=C2O)C=C(S(=O...,PNO2,O=[N+]([O-])c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(...
3,1.163151,2.186051,0.000000,-1.309333,-2.302585,-1.272966,-0.820981,0.405465,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP1,O=C(O)Cc1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)c...
4,0.741937,1.791759,0.182322,-1.171183,-1.660731,-0.867501,-0.430783,0.587787,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP3,CC(C)(C)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)...
5,1.335001,2.079442,0.182322,-0.776529,-1.660731,-0.891598,0.182322,1.163151,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP4,Cc1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc(c2O)...
6,1.458615,2.041220,0.262364,-0.820981,-1.966113,-1.171183,-0.510826,0.470004,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP5,O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...
7,0.470004,-0.127833,-2.120264,-4.342806,-4.199705,-3.244194,-3.015935,-1.469676,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP6,O=C(O)C[C@H](NC(=O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(...
8,0.587787,0.336472,-0.579818,-2.975930,-2.617296,-4.017384,-2.040221,-0.342490,OC1=C(C=C(C=C1CC2=C(C(C3)=CC(S(=O)(O)=O)=C2)O)...,AP7,NC(=O)[C@H](CC(=O)O)NC(=O)c1ccc(-c2cc3c(O)c(c2...
9,-1.309333,0.000000,-3.079114,-5.184989,-5.020686,-3.772261,-4.199705,-2.603690,OC1=C(CC2=CC(S(=O)(O)=O)=CC(CC3=CC(S(=O)(O)=O)...,AP8,O=C(N[C@@H](Cc1ccc(O)cc1)C(=O)O)c1ccc(-c2cc3c(...


In [14]:
import pickle

# Assuming your dictionary is named 'results' and the specific dictionary you want to save is at index 5
dictionary_to_save = results[5]

# Specify the file name where you want to save the pickle file
file_name = "./GCN_regression.pkl"

# Open the file in binary write mode and save the dictionary
with open(file_name, "wb") as file:
    pickle.dump(dictionary_to_save, file)

print(f"Dictionary saved as {file_name}")

Dictionary saved as ./GCN_regression.pkl


In [14]:
import pickle

# Load the contents of the classification.pkl dictionary
try:
    with open('/notebooks/GNN/GCN_regression.pkl', 'rb') as file:
        classification_dict = pickle.load(file)
    
    # Display the contents
    print(classification_dict)
    
    # If it's a large dictionary, you might want to explore it in parts:
    print("Keys:", classification_dict.keys())
    
    # Print the first few items (if it's a large dictionary)
    for key, value in list(classification_dict.items())[:5]:
        print(f"{key}: {value}")

except FileNotFoundError:
    print("The file 'classification.pkl' was not found in the current directory.")
except Exception as e:
    print(f"An error occurred: {e}")


{'AP8': {'H3K4': {'predicted': 1.0213838, 'actual': -1.3093333}, 'H3K4ac': {'predicted': 1.37591, 'actual': 0.0}, 'H3K4me1': {'predicted': -0.030764328, 'actual': -3.079114}, 'H3K4me2': {'predicted': -1.2975001, 'actual': -5.1849885}, 'H3K4me3': {'predicted': -1.6570596, 'actual': -5.0206857}, 'H3K9me3': {'predicted': -0.9378937, 'actual': -3.7722611}, 'H3R2me2a': {'predicted': -0.41601068, 'actual': -4.199705}, 'H3R2me2s': {'predicted': 0.23874505, 'actual': -2.6036901}}, 'AH4': {'H3K4': {'predicted': 0.7992573, 'actual': -0.116533816}, 'H3K4ac': {'predicted': 1.0970049, 'actual': 1.410987}, 'H3K4me1': {'predicted': -0.14709036, 'actual': -2.207275}, 'H3K4me2': {'predicted': -1.353481, 'actual': -5.2983174}, 'H3K4me3': {'predicted': -1.5851107, 'actual': -3.506558}, 'H3K9me3': {'predicted': -0.88736695, 'actual': -2.207275}, 'H3R2me2a': {'predicted': -0.49689317, 'actual': -2.3330443}, 'H3R2me2s': {'predicted': 0.042138807, 'actual': -1.4271164}}, 'AH3': {'H3K4': {'predicted': 0.73924

In [16]:
overall_r2 = calculate_overall_r2(classification_dict)
overall_r2

0.2725092090293668